# Prepare Additional Data for Embedding

Normalizes four additional data sources into Lambda-compatible format:
- **NYT_filtered_500**: `combined_text` → `text`, `_id` → `identifier`
- **Economist**: `ocr_text` → `text`, `article_id` → `identifier` (concat weekly files per year)
- **FT**: `text_cleaned` → `text`, `id` → `identifier` (filter `ocr_score >= 50`)
- **Newswire**: `cleaned_article` → `text`, index-based `identifier` (JSON → parquet)

**Output:**
- Staging: `D:\additional_data\staging\{collection}\{year}\*.parquet`
- Tar archives: `D:\temp\year_{collection}{year}.tar` (for Lambda upload)

In [1]:
import pandas as pd
import json
import subprocess
import shutil
import re
from pathlib import Path
from tqdm import tqdm

# --- CONFIG ---
PROJECT_ROOT = Path(r"C:\Users\danielyoon\Dropbox\hist_LLM")
STAGING_DIR = Path(r"D:\additional_data\staging")
TAR_DIR = Path(r"D:\temp")
TAR_DIR.mkdir(parents=True, exist_ok=True)

# Max chars to keep (BGE-large truncates at 512 tokens ≈ 2000 chars;
# using 3000 as safe upper bound, matching Embed_All_Data_Local.ipynb)
MAX_CHARS = 3000

# Source definitions
SOURCES = {
    "nyt": {
        "dir": PROJECT_ROOT / "Data" / "additional_data" / "news_archives" / "NYT_filtered_500char",
        "text_col": "combined_text",
        "id_col": "_id",
    },
    "economist": {
        "dir": PROJECT_ROOT / "Data" / "additional_data" / "news_archives" / "Economist",
        "text_col": "ocr_text",
        "id_col": "article_id",
    },
    "ft": {
        "dir": PROJECT_ROOT / "Data" / "additional_data" / "news_archives" / "FT",
        "text_col": "text_cleaned",
        "id_col": "id",
        "ocr_score_min": 50,
    },
    "newswire": {
        "dir": PROJECT_ROOT / "Data" / "additional_data" / "newswire",
        "text_col": "cleaned_article",
        "id_col": None,  # index-based
    },
}

print("Config loaded. Sources:")
for name, cfg in SOURCES.items():
    exists = cfg["dir"].exists()
    print(f"  {name}: {cfg['dir']} [{'OK' if exists else 'MISSING'}]")

Config loaded. Sources:
  nyt: C:\Users\danielyoon\Dropbox\hist_LLM\Data\additional_data\news_archives\NYT_filtered_500char [OK]
  economist: C:\Users\danielyoon\Dropbox\hist_LLM\Data\additional_data\news_archives\Economist [OK]
  ft: C:\Users\danielyoon\Dropbox\hist_LLM\Data\additional_data\news_archives\FT [OK]
  newswire: C:\Users\danielyoon\Dropbox\hist_LLM\Data\additional_data\newswire [OK]


## Helper Functions

In [2]:
def extract_year_from_filename(filename, source):
    """Extract year from filename based on source naming convention."""
    name = Path(filename).stem
    if source == "nyt":
        # nyt_1851.parquet → 1851
        return int(name.split("_")[1])
    elif source == "economist":
        # economist_1843-0801.parquet → 1843
        return int(name.split("_")[1].split("-")[0])
    elif source == "ft":
        # 1888.parquet → 1888
        return int(name)
    elif source == "newswire":
        # 1878_data_clean.json → 1878
        return int(name.split("_")[0])
    return None


def normalize_dataframe(df, source, cfg, year, file_idx=0):
    """Normalize a dataframe to [identifier, text] format."""
    text_col = cfg["text_col"]
    id_col = cfg["id_col"]
    
    # Filter FT by OCR score
    if source == "ft" and "ocr_score_min" in cfg:
        if "ocr_score" in df.columns:
            before = len(df)
            df = df[pd.to_numeric(df["ocr_score"], errors="coerce") >= cfg["ocr_score_min"]].copy()
            if before > 0:
                pct = 100 * len(df) / before
                # Only print if significant filtering occurred
                if pct < 95:
                    print(f"    FT OCR filter: {before} → {len(df)} ({pct:.0f}% kept)")
    
    # Check text column exists
    if text_col not in df.columns:
        print(f"    [WARN] Column '{text_col}' not found. Available: {list(df.columns)}")
        return None
    
    # Create identifier
    if id_col and id_col in df.columns:
        identifiers = df[id_col].astype(str).values
    else:
        identifiers = [f"{source}_{year}_{file_idx}_{i}" for i in range(len(df))]
    
    # Build normalized dataframe
    result = pd.DataFrame({
        "identifier": identifiers,
        "text": df[text_col].astype(str).str[:MAX_CHARS].values,
    })
    
    # Drop rows with empty/null text
    result = result[result["text"].str.strip().str.len() > 0].reset_index(drop=True)
    
    return result


def save_and_tar(df, source, year):
    """Save normalized parquet to staging and create tar archive."""
    # Save staging parquet
    staging_dir = STAGING_DIR / source / str(year)
    staging_dir.mkdir(parents=True, exist_ok=True)
    staging_path = staging_dir / f"{source}_{year}.parquet"
    df.to_parquet(staging_path, index=False)
    
    # Create tar archive
    tar_name = f"year_{source}{year}.tar"
    tar_path = TAR_DIR / tar_name
    subprocess.run(
        ["tar", "-cf", str(tar_path), "-C", str(staging_dir), "."],
        check=True, capture_output=True
    )
    
    tar_size_mb = tar_path.stat().st_size / 1024 / 1024
    return tar_path, tar_size_mb


print("Helpers loaded.")

Helpers loaded.


## Process NYT_filtered_500

In [3]:
source = "nyt"
cfg = SOURCES[source]
files = sorted(cfg["dir"].glob("nyt_*.parquet"))
print(f"Found {len(files)} NYT files")

nyt_stats = []
for f in tqdm(files, desc="NYT"):
    year = extract_year_from_filename(f.name, source)
    
    # Skip if tar already exists
    tar_path = TAR_DIR / f"year_{source}{year}.tar"
    if tar_path.exists():
        continue
    
    df = pd.read_parquet(f)
    normalized = normalize_dataframe(df, source, cfg, year)
    if normalized is None or len(normalized) == 0:
        continue
    
    _, tar_mb = save_and_tar(normalized, source, year)
    nyt_stats.append({"year": year, "rows": len(normalized), "tar_mb": tar_mb})

print(f"\nNYT complete: {len(nyt_stats)} years processed")
if nyt_stats:
    total_rows = sum(s["rows"] for s in nyt_stats)
    total_mb = sum(s["tar_mb"] for s in nyt_stats)
    print(f"  Total rows: {total_rows:,}")
    print(f"  Total tar size: {total_mb:.1f} MB")

Found 167 NYT files


NYT: 100%|██████████| 167/167 [06:55<00:00,  2.49s/it]


NYT complete: 167 years processed
  Total rows: 2,805,408
  Total tar size: 1314.1 MB


## Process Economist

In [4]:
source = "economist"
cfg = SOURCES[source]
all_files = sorted(cfg["dir"].glob("economist_*.parquet"))
print(f"Found {len(all_files)} Economist files")

# Group files by year
files_by_year = {}
for f in all_files:
    year = extract_year_from_filename(f.name, source)
    if year not in files_by_year:
        files_by_year[year] = []
    files_by_year[year].append(f)

print(f"Years: {min(files_by_year)}-{max(files_by_year)} ({len(files_by_year)} years)")

econ_stats = []
for year in tqdm(sorted(files_by_year.keys()), desc="Economist"):
    # Skip if tar already exists
    tar_path = TAR_DIR / f"year_{source}{year}.tar"
    if tar_path.exists():
        continue
    
    # Concat all weekly files for this year
    year_dfs = []
    for f in files_by_year[year]:
        try:
            df = pd.read_parquet(f)
            year_dfs.append(df)
        except Exception as e:
            print(f"  [WARN] Error reading {f.name}: {e}")
    
    if not year_dfs:
        continue
    
    combined = pd.concat(year_dfs, ignore_index=True)
    normalized = normalize_dataframe(combined, source, cfg, year)
    if normalized is None or len(normalized) == 0:
        continue
    
    _, tar_mb = save_and_tar(normalized, source, year)
    econ_stats.append({"year": year, "rows": len(normalized), "tar_mb": tar_mb})

print(f"\nEconomist complete: {len(econ_stats)} years processed")
if econ_stats:
    total_rows = sum(s["rows"] for s in econ_stats)
    total_mb = sum(s["tar_mb"] for s in econ_stats)
    print(f"  Total rows: {total_rows:,}")
    print(f"  Total tar size: {total_mb:.1f} MB")

Found 8891 Economist files
Years: 1843-2014 (172 years)


Economist: 100%|██████████| 172/172 [5:01:13<00:00, 105.08s/it]  


Economist complete: 172 years processed
  Total rows: 926,029
  Total tar size: 1194.7 MB


## Process FT (Financial Times)

In [5]:
source = "ft"
cfg = SOURCES[source]
# FT files are named {year}.parquet — filter to only numeric filenames
files = sorted([f for f in cfg["dir"].glob("*.parquet") if f.stem.isdigit()])
print(f"Found {len(files)} FT files")

ft_stats = []
for f in tqdm(files, desc="FT"):
    year = extract_year_from_filename(f.name, source)
    
    # Skip if tar already exists
    tar_path = TAR_DIR / f"year_{source}{year}.tar"
    if tar_path.exists():
        continue
    
    df = pd.read_parquet(f)
    normalized = normalize_dataframe(df, source, cfg, year)
    if normalized is None or len(normalized) == 0:
        continue
    
    _, tar_mb = save_and_tar(normalized, source, year)
    ft_stats.append({"year": year, "rows": len(normalized), "tar_mb": tar_mb})

print(f"\nFT complete: {len(ft_stats)} years processed")
if ft_stats:
    total_rows = sum(s["rows"] for s in ft_stats)
    total_mb = sum(s["tar_mb"] for s in ft_stats)
    print(f"  Total rows: {total_rows:,}")
    print(f"  Total tar size: {total_mb:.1f} MB")

Found 119 FT files


FT:   0%|          | 0/119 [00:00<?, ?it/s]

    FT OCR filter: 21625 → 18439 (85% kept)


FT:   1%|          | 1/119 [00:03<06:22,  3.24s/it]

    FT OCR filter: 28413 → 24413 (86% kept)


...

    FT OCR filter: 310663 → 291400 (94% kept)


FT:  99%|█████████▉| 118/119 [14:00<00:12, 12.40s/it]

    FT OCR filter: 253789 → 236551 (93% kept)


FT: 100%|██████████| 119/119 [14:15<00:00,  7.19s/it]


FT complete: 119 years processed
  Total rows: 14,371,144
  Total tar size: 11388.1 MB


## Process Newswire

In [6]:
source = "newswire"
cfg = SOURCES[source]
files = sorted(cfg["dir"].glob("*_data_clean.json"))
print(f"Found {len(files)} Newswire files")

nw_stats = []
for f in tqdm(files, desc="Newswire"):
    year = extract_year_from_filename(f.name, source)
    
    # Skip if tar already exists
    tar_path = TAR_DIR / f"year_{source}{year}.tar"
    if tar_path.exists():
        continue
    
    # Load JSON
    try:
        with open(f, "r", encoding="utf-8") as fp:
            data = json.load(fp)
    except Exception as e:
        print(f"  [WARN] Error reading {f.name}: {e}")
        continue
    
    df = pd.DataFrame(data)
    normalized = normalize_dataframe(df, source, cfg, year)
    if normalized is None or len(normalized) == 0:
        continue
    
    _, tar_mb = save_and_tar(normalized, source, year)
    nw_stats.append({"year": year, "rows": len(normalized), "tar_mb": tar_mb})

print(f"\nNewswire complete: {len(nw_stats)} years processed")
if nw_stats:
    total_rows = sum(s["rows"] for s in nw_stats)
    total_mb = sum(s["tar_mb"] for s in nw_stats)
    print(f"  Total rows: {total_rows:,}")
    print(f"  Total tar size: {total_mb:.1f} MB")

Found 100 Newswire files


Newswire:  80%|████████  | 80/100 [05:35<01:56,  5.80s/it]

  [WARN] Error reading 1957_data_clean.json: Unterminated string starting at: line 14791963 column 28 (char 435699074)


Newswire: 100%|██████████| 100/100 [07:29<00:00,  4.49s/it]


Newswire complete: 99 years processed
  Total rows: 2,668,458
  Total tar size: 1647.6 MB


## Summary

In [7]:
# Count total tar files ready for upload
all_tars = sorted(TAR_DIR.glob("year_*.tar"))
print(f"Total tar archives ready for upload: {len(all_tars)}")
print()

for source_code in ["nyt", "economist", "ft", "newswire"]:
    source_tars = [t for t in all_tars if t.name.startswith(f"year_{source_code}")]
    if source_tars:
        total_mb = sum(t.stat().st_size / 1024 / 1024 for t in source_tars)
        print(f"  {source_code}: {len(source_tars)} files, {total_mb:.1f} MB total")

Total tar archives ready for upload: 557

  nyt: 167 files, 1314.1 MB total
  economist: 172 files, 1194.7 MB total
  ft: 119 files, 11388.1 MB total
  newswire: 99 files, 1647.6 MB total


## Sanity Check
Load one staging parquet to verify format matches what Lambda expects.

In [8]:
# Pick a random staging file to inspect
staging_files = list(STAGING_DIR.rglob("*.parquet"))
if staging_files:
    sample = pd.read_parquet(staging_files[0])
    print(f"Sample file: {staging_files[0]}")
    print(f"Columns: {list(sample.columns)}")
    print(f"Rows: {len(sample)}")
    print(f"\nFirst row text (first 200 chars):")
    print(sample.iloc[0]['text'][:200])
    print(f"\nFirst row identifier: {sample.iloc[0]['identifier']}")
else:
    print("No staging files found yet.")

Sample file: D:\additional_data\staging\nyt\1851\nyt_1851.parquet
Columns: ['identifier', 'text']
Rows: 209

First row text (first 200 chars):
NEW-YORK CITY.; AMUSEMENTS THIS EVENING. AN INTERESTING REVIEW. THE NEW-YORK HOSPITAL. THE EXHIBITION AT CASTLE GARDEN. A WEDDING AT THE CITY HALL. DEATH OF OFFICER HUTHWAITR. CONSECRATED BURYING GROU

First row identifier: 4fbfd28145c1498b0d005fa6
